# Dragon Fruit Plant Image Optimizer Study 

This notebook implements the Dragon Fruit binary classification experiment with VGG19 and EfficientNetB3 active.

Project setup retained:
- Binary classification: `comdoença` / `Comdoenca` = with disease, `semdoença` / `Endoenca` = without disease
- Input size: 256 × 256
- Batch size: 32
- Epochs: 50
- Dense head: 256 → 128 → 64
- Dropout: 0.6 → 0.4 → 0.3
- Optimizers: Adam, Adagrad, Adamax, Adadelta, SGD, RMSprop
- Learning rates: 0.0001, 0.00001, 0.000001


In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import unicodedata

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.applications import VGG19, EfficientNetB3, ResNet50, DenseNet121
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


In [ ]:
DATASET_DIR = Path("Dataset")
TRAIN_DIR = DATASET_DIR / "train"
VAL_DIR = DATASET_DIR / "validation"
TEST_DIR = DATASET_DIR / "test"

IMG_SIZE = (256, 256)
BATCH_SIZE = 32
EPOCHS = 50

SUMMARY_DIR = Path("outputs_summaries")
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

def get_output_dirs(model_name):
    output_dir = Path(f"outputs_{model_name}")
    plots_dir = output_dir / "plots"
    reports_dir = output_dir / "reports"
    confusion_dir = output_dir / "confusion_matrices"

    output_dir.mkdir(parents=True, exist_ok=True)
    plots_dir.mkdir(parents=True, exist_ok=True)
    reports_dir.mkdir(parents=True, exist_ok=True)
    confusion_dir.mkdir(parents=True, exist_ok=True)

    return output_dir, plots_dir, reports_dir, confusion_dir

for required_dir in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    if not required_dir.exists():
        raise FileNotFoundError(f"Missing directory: {required_dir}")


In [ ]:
for dirname, _, filenames in os.walk(DATASET_DIR):
    for filename in filenames[:5]:
        print(os.path.join(dirname, filename))


## Data Generators

Training uses augmentation and normalization. Validation and test use normalization only. Test is set to `shuffle=False` so predictions line up with `test.classes`.


In [ ]:
image_generatorFullAugment = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.15,
    shear_range=0.3,
    zoom_range=0.30,
    horizontal_flip=True,
    samplewise_center=True,
    samplewise_std_normalization=True,
)

image_generatorNoAugment = ImageDataGenerator(
    samplewise_center=True,
    samplewise_std_normalization=True,
)


In [ ]:
train = image_generatorFullAugment.flow_from_directory(
    TRAIN_DIR,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    target_size=IMG_SIZE,
    shuffle=True,
    seed=SEED,
)

validation = image_generatorNoAugment.flow_from_directory(
    VAL_DIR,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    target_size=IMG_SIZE,
    shuffle=False,
)

test = image_generatorNoAugment.flow_from_directory(
    TEST_DIR,
    batch_size=1,
    class_mode="binary",
    target_size=IMG_SIZE,
    shuffle=False,
)

print("Class indices:", train.class_indices)


## Experiment Settings


In [ ]:
LEARNING_RATES = [0.0001, 0.00001, 0.000001]
OPTIMIZERS = ["adam", "adagrad", "adamax", "adadelta", "sgd", "rmsprop"]

MODELS_TO_RUN = [
    "vgg19",
    "efficientnetb3",
    # "resnet50",
    # "densenet121",
]

POSITIVE_CLASS_NAMES = [
    "Comdoenca",
    "comdoenca",
    "comdoença",
    "With Disease",
    "with disease",
]


def normalize_class_name(name):
    name = name.lower().strip()
    name = unicodedata.normalize("NFKD", name)
    name = "".join(char for char in name if not unicodedata.combining(char))
    return name


def get_positive_class_index(generator):
    normalized_positive_names = [
        normalize_class_name(name) for name in POSITIVE_CLASS_NAMES
    ]

    for class_name, class_index in generator.class_indices.items():
        if normalize_class_name(class_name) in normalized_positive_names:
            return class_index

    raise ValueError(
        f"Positive disease class was not found. Available classes: {generator.class_indices}"
    )


def get_disease_scores(raw_prob_class_1, positive_class_index):
    if positive_class_index == 1:
        return raw_prob_class_1

    return 1.0 - raw_prob_class_1


In [ ]:
def get_optimizer(optimizer_name, lr):
    optimizer_name = optimizer_name.lower()

    optimizers = {
        "adam": tf.keras.optimizers.Adam,
        "adagrad": tf.keras.optimizers.Adagrad,
        "adamax": tf.keras.optimizers.Adamax,
        "adadelta": tf.keras.optimizers.Adadelta,
        "sgd": tf.keras.optimizers.SGD,
        "rmsprop": tf.keras.optimizers.RMSprop,
    }

    if optimizer_name not in optimizers:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

    return optimizers[optimizer_name](learning_rate=lr)


def get_base_model(model_name):
    model_name = model_name.lower()

    if model_name == "vgg19":
        return VGG19(include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,))

    if model_name == "efficientnetb3":
        return EfficientNetB3(include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,))

    if model_name == "resnet50":
        return ResNet50(include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,))

    if model_name == "densenet121":
        return DenseNet121(include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,))

    raise ValueError(f"Unsupported model: {model_name}")


def build_model(model_name, lr, optimizer_name):
    tf.keras.backend.clear_session()

    base_model = get_base_model(model_name)
    base_model.trainable = False

    model = Sequential([
        base_model,
        GlobalAveragePooling2D(),
        Dense(256, activation="relu"),
        BatchNormalization(),
        Dropout(0.6),
        Dense(128, activation="relu"),
        BatchNormalization(),
        Dropout(0.4),
        Dense(64, activation="relu"),
        BatchNormalization(),
        Dropout(0.3),
        Dense(1, activation="sigmoid"),
    ])

    model.compile(
        optimizer=get_optimizer(optimizer_name, lr),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.AUC(name="auc"),
        ],
    )

    return model


In [ ]:
def safe_divide(numerator, denominator):
    return numerator / denominator if denominator != 0 else 0.0


def evaluate_predictions(test_generator, raw_prob_class_1, model_name, optimizer_name, lr):
    positive_class_index = get_positive_class_index(test_generator)

    y_true = (test_generator.classes == positive_class_index).astype(int)
    y_score = get_disease_scores(raw_prob_class_1, positive_class_index)
    y_pred = (y_score >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    metrics = {
        "model": model_name,
        "optimizer": optimizer_name,
        "learning_rate": lr,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "specificity": safe_divide(tn, tn + fp),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, y_score),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

    report = classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["Without Disease", "With Disease"],
        output_dict=True,
        zero_division=0,
    )

    return metrics, pd.DataFrame(report).transpose(), y_true, y_score, y_pred


In [ ]:
def plot_training_history(history, run_name, plots_dir):
    history_dict = history.history

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0, 0].plot(history_dict.get("loss", []), label="Loss")
    axes[0, 0].plot(history_dict.get("val_loss", []), label="Val_Loss")
    axes[0, 0].set_title("Loss Evolution")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].set_ylabel("Loss")
    axes[0, 0].legend()

    axes[0, 1].plot(history_dict.get("accuracy", []), label="Accuracy")
    axes[0, 1].plot(history_dict.get("val_accuracy", []), label="Val_Accuracy")
    axes[0, 1].set_title("Accuracy Evolution")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].set_ylabel("Accuracy")
    axes[0, 1].legend()

    axes[1, 0].plot(history_dict.get("auc", []), label="AUC")
    axes[1, 0].plot(history_dict.get("val_auc", []), label="Val_AUC")
    axes[1, 0].set_title("AUC Evolution")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].set_ylabel("AUC")
    axes[1, 0].legend()

    axes[1, 1].axis("off")

    fig.suptitle(run_name)
    fig.tight_layout()

    path = plots_dir / f"{run_name}_history.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    return path

def plot_roc_curve(y_true, y_score, run_name, plots_dir):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    auc_value = roc_auc_score(y_true, y_score)

    fig = plt.figure(figsize=(7, 6))
    plt.plot(fpr, tpr, label=f"AUC = {auc_value:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--", label="Random")
    plt.title(f"ROC Curve — {run_name}")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    plt.tight_layout()

    path = plots_dir / f"{run_name}_roc.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    return path


In [ ]:
def run_experiment(model_name, optimizer_name, lr):
    output_dir, plots_dir, reports_dir, confusion_dir = get_output_dirs(model_name)

    run_name = f"{model_name}_{optimizer_name}_lr_{lr:g}".replace(".", "p")
    print()
    print(f"Running: {run_name}")

    model = build_model(model_name, lr, optimizer_name)

    history = model.fit(
        train,
        epochs=EPOCHS,
        validation_data=validation,
        verbose=1,
    )

    test.reset()
    raw_prob_class_1 = model.predict(test, verbose=0).ravel()

    metrics, report_df, y_true, y_score, y_pred = evaluate_predictions(
        test,
        raw_prob_class_1,
        model_name,
        optimizer_name,
        lr,
    )

    history_plot_path = plot_training_history(history, run_name, plots_dir)
    roc_plot_path = plot_roc_curve(y_true, y_score, run_name, plots_dir)

    report_path = reports_dir / f"{run_name}_classification_report.csv"
    report_df.to_csv(report_path)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    confusion_df = pd.DataFrame(
        cm,
        index=["Actual Without Disease", "Actual With Disease"],
        columns=["Predicted Without Disease", "Predicted With Disease"],
    )

    confusion_path = confusion_dir / f"{run_name}_confusion_matrix.csv"
    confusion_df.to_csv(confusion_path)

    metrics["history_plot"] = str(history_plot_path)
    metrics["roc_plot"] = str(roc_plot_path)
    metrics["classification_report"] = str(report_path)
    metrics["confusion_matrix"] = str(confusion_path)

    print("Confusion matrix using With Disease as positive class:")
    display(confusion_df)
    display(pd.DataFrame([metrics]))
    display(report_df)

    return metrics

In [ ]:
all_results = []

def get_run_name(model_name, optimizer_name, lr):
    return f"{model_name}_{optimizer_name}_lr_{lr:g}".replace(".", "p")


def run_outputs_exist(model_name, optimizer_name, lr):
    output_dir, plots_dir, reports_dir, confusion_dir = get_output_dirs(model_name)
    run_name = get_run_name(model_name, optimizer_name, lr)

    expected_files = [
        plots_dir / f"{run_name}_history.png",
        plots_dir / f"{run_name}_roc.png",
        reports_dir / f"{run_name}_classification_report.csv",
        confusion_dir / f"{run_name}_confusion_matrix.csv",
    ]

    missing_files = [file for file in expected_files if not file.exists()]

    if missing_files:
        print(f"Incomplete or missing run, will run: {run_name}")
        for file in missing_files:
            print(f"Missing: {file}")
        return False

    return True


for model_name in MODELS_TO_RUN:
    model_results = []

    output_dir, plots_dir, reports_dir, confusion_dir = get_output_dirs(model_name)

    for optimizer_name in OPTIMIZERS:
        for lr in LEARNING_RATES:
            run_name = get_run_name(model_name, optimizer_name, lr)

            if run_outputs_exist(model_name, optimizer_name, lr):
                print(f"Skipping completed run: {run_name}")
                continue

            result = run_experiment(model_name, optimizer_name, lr)
            model_results.append(result)
            all_results.append(result)

            partial_results_df = pd.DataFrame(model_results)
            partial_results_df.to_csv(
                output_dir / f"{model_name}_partial_results_summary.csv",
                index=False,
            )

    if model_results:
        model_results_df = pd.DataFrame(model_results)
        model_results_df.to_csv(
            output_dir / f"{model_name}_new_results_summary.csv",
            index=False,
        )
        display(model_results_df)

all_results_df = pd.DataFrame(all_results)

if not all_results_df.empty:
    combined_path = SUMMARY_DIR / "all_new_results_summary.csv"
    all_results_df.to_csv(combined_path, index=False)
    print(f"Saved combined new results to: {combined_path}")
    display(all_results_df)
else:
    print("No new runs were performed. All expected output files already exist.")


## Best Run Summary

This summary uses only the runs performed in the current session. If all runs were skipped because output files already exist, no new summary table will be displayed here.


In [ ]:
if "all_results_df" in globals() and not all_results_df.empty:
    best_by_auc = all_results_df.sort_values("auc", ascending=False).head(1)
    best_by_accuracy = all_results_df.sort_values("accuracy", ascending=False).head(1)

    print("Best new run by AUC:")
    display(best_by_auc)

    print("Best new run by Accuracy:")
    display(best_by_accuracy)
else:
    print("No new results in this session. Check the model-specific output folders for existing completed runs.")
